In [1]:
#| echo: false
#| warning: false
#| output: asis

from IPython.display import Markdown, display
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.formula.api import logit, probit
from scipy.stats import norm, logistic, chi2
import warnings
warnings.simplefilter(action='ignore', category=Warning)

# Загрузка данных
loanapp = pd.read_csv('../datasets/loanapp.csv')
mroz_Greene = pd.read_csv('../datasets/TableF5-1.csv')
SwissLabor = pd.read_csv('../datasets/SwissLabor.csv')

# Очистка пропусков для loanapp
loanapp = loanapp[['approve', 'appinc', 'mortno', 'unem', 'dep', 'male', 'married', 'yjob', 'self']].dropna()

# Конвертация participation для SwissLabor
SwissLabor['participation_num'] = (SwissLabor['participation'] == 'yes').astype(int)

# Настройки графиков
plt.rcParams['figure.figsize'] = (8.5, 6)
plt.rcParams['font.size'] = 10

# --- ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ДЛЯ QUARTO ---
def display_answer(text, use_callout=True, theme="note", title="Критическое значение"):
    if use_callout:
        if title:
            output_md = f"\n\n::: {{.callout-{theme} title=\"{title}\" icon=\"true\"}}\n{text}\n:::\n\n"
        else:
            output_md = f"\n\n::: {{.callout-{theme} icon=\"true\"}}\n{text}\n:::\n\n"
    else:
        if title:
            output_md = f"\n\n**{title}:** {text}\n\n"
        else:
            output_md = f"\n\n{text}\n\n"
    display(Markdown(output_md))

def print_coefs(res, digits=3):
    df = pd.DataFrame({'Переменная': res.params.index, 'Оценка (MLE)': np.round(res.params, digits)})
    display(Markdown(df.to_markdown(index=False)))

def get_stars(p_value):
    if p_value < 0.01: return '***'
    elif p_value < 0.05: return '**'
    elif p_value < 0.1: return '*'
    return ''

def print_z_test(res, sign_level=0.05, digits=4):
    df = pd.DataFrame({
        'Coef.': res.params,
        'Std.Err.': res.bse,
        'z-value': res.tvalues,
        'p-value': res.pvalues
    }).round(digits)
    df['Signif.'] = df['p-value'].apply(get_stars)
    for const_name in ['const', 'Intercept']:
        if const_name in df.index:
            df.loc[const_name, 'Signif.'] = ''
    legend = "\n*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$"
    display(Markdown(df.to_markdown(index=True) + "\n" + legend))

def print_significant_coeffs(res, sign_level=0.05):
    all_sign_coeffs = res.params.index[res.pvalues < sign_level].tolist()
    sign_coeffs = [coef for coef in all_sign_coeffs if coef not in ['const', 'Intercept']]
    percent_level = f"{sign_level * 100:g}"
    title = f"Коэффициенты, значимые на уровне {percent_level}%"
    if sign_coeffs:
        text = ", ".join([f"`{coef}`" for coef in sign_coeffs])
    else:
        text = "Значимых коэффициентов не выявлено."
    display_answer(text, use_callout=True, theme="note", title=title)


# Нормальное и логистическое распределения

**Стандартное нормальное (гауссово) распределение $N(0, 1)$**

* Плотность вероятности:
$$\phi(t) = \frac{1}{\sqrt{2\pi}} \exp\left(-\frac{t^2}{2}\right), \quad t \in \mathbb{R}$$

* (Кумулятивная) функция распределения:
$$\Phi(x) = \int\limits_{-\infty}^x \phi(t) \, dt, \quad x \in \mathbb{R}$$

---

**Логистическое распределение**

* (Кумулятивная) функция распределения:
$$\Lambda(x) = \frac{\exp(x)}{1 + \exp(x)}, \quad x \in \mathbb{R}$$

* Плотность вероятности:
$$\lambda(x) = \Lambda'(x) = \frac{\exp(x)}{(1 + \exp(x))^2}, \quad x \in \mathbb{R}$$

---

Плотности на одном графике

In [2]:
#| echo: false
#| warning: false

# Настройка приятного стиля по умолчанию
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 14})

# Общие параметры
x = np.linspace(-6, 6, 500)
p = np.linspace(0.001, 0.999, 500) # Избегаем 0 и 1 для PPF
color_norm = '#1f77b4' # Синий
color_log = '#d62728'  # Красный

# ==========================================
# 1. Функция плотности (PDF)
# ==========================================
plt.figure(figsize=(8, 5))
plt.plot(x, norm.pdf(x), color=color_norm, lw=2.5, label=r'Нормальное: $\phi(x)$')
plt.plot(x, logistic.pdf(x), color=color_log, lw=2.5, linestyle='--', label=r'Логистическое: $\lambda(x)$')

plt.axvline(0, color='black', linestyle=':', alpha=0.5)
plt.title('Функция плотности (PDF)')
plt.ylabel('Плотность вероятности')
plt.xlabel('Значение x')
plt.legend(frameon=True, shadow=True)
plt.tight_layout()
plt.show()

<Figure size 2400x1500 with 1 Axes>

Функции распределения на одном графике

In [3]:
#| echo: false
#| warning: false


# Настройка приятного стиля по умолчанию
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 14})

# Общие параметры
x = np.linspace(-6, 6, 500)
p = np.linspace(0.001, 0.999, 500) # Избегаем 0 и 1 для PPF
color_norm = '#1f77b4' # Синий
color_log = '#d62728'  # Красный

# ==========================================
# 2. Кумулятивная функция (CDF)
# ==========================================
plt.figure(figsize=(8, 5))
plt.plot(x, norm.cdf(x), color=color_norm, lw=2.5, label=r'Нормальное: $\Phi(x)$')
plt.plot(x, logistic.cdf(x), color=color_log, lw=2.5, linestyle='--', label=r'Логистическое: $\Lambda(x)$')

plt.axvline(0, color='black', linestyle=':', alpha=0.5)
plt.axhline(0.5, color='black', linestyle=':', alpha=0.5) # Полезная линия для медианы
plt.title('Кумулятивная функция (CDF)')
plt.ylabel('Вероятность')
plt.xlabel('Значение x')
plt.legend(frameon=True, shadow=True)
plt.tight_layout()
plt.show()

<Figure size 2400x1500 with 1 Axes>

Обратные функции распределения 

$$
logit(p)=\Lambda^{-1}(p)=\log\frac{p}{1-p} \quad \text{ и } \quad probit(p)=\Phi^{-1}(p), \quad 0<p<1
$$

Их графики

In [4]:
#| echo: false
#| warning: false


# Настройка приятного стиля по умолчанию
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'axes.titlesize': 14})

# Общие параметры
x = np.linspace(-6, 6, 500)
p = np.linspace(0.001, 0.999, 500) # Избегаем 0 и 1 для PPF
color_norm = '#1f77b4' # Синий
color_log = '#d62728'  # Красный

# ==========================================
# 3. Квантильная функция (PPF)
# ==========================================
plt.figure(figsize=(8, 5))
plt.plot(p, norm.ppf(p), color=color_norm, lw=2.5, label='Probit (нормальное)')
plt.plot(p, logistic.ppf(p), color=color_log, lw=2.5, linestyle='--', label='Logit (логистическое)')

plt.axhline(0, color='black', linestyle=':', alpha=0.5)
plt.axvline(0.5, color='black', linestyle=':', alpha=0.5)
plt.ylim(-6, 6)
plt.title('Квантильная функция (PPF)')
plt.ylabel('Значение (квантиль)')
plt.xlabel('Вероятность p')
plt.legend(frameon=True, shadow=True)
plt.tight_layout()
plt.show()

<Figure size 2400x1500 with 1 Axes>

In [5]:
#| echo: false
#| warning: false

# 1. Настройка приятного стиля по умолчанию
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 11, 'axes.titlesize': 13})

x = np.linspace(-6, 6, 500)
# Для PPF лучше избегать точных 0 и 1, чтобы не получить бесконечность на краях графика
p = np.linspace(0.001, 0.999, 500) 

# 2. Создаем одну фигуру с 3 графиками в ряд (1 строка, 3 колонки)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Определяем базовые цвета для единообразия
color_norm = '#1f77b4' # Глубокий синий
color_log = '#d62728'  # Приглушенный красный

# --- График 1: Функция плотности (PDF) ---
axes[0].plot(x, norm.pdf(x), color=color_norm, lw=2.5, label=r'Нормальное: $\phi(x)$')
axes[0].plot(x, logistic.pdf(x), color=color_log, lw=2.5, linestyle='--', label=r'Логистическое: $\lambda(x)$')
axes[0].set_title('Функция плотности (PDF)')
axes[0].set_xlabel('$x$')
axes[0].set_ylabel('Плотность')

# --- График 2: Кумулятивная функция (CDF) ---
axes[1].plot(x, norm.cdf(x), color=color_norm, lw=2.5, label=r'Нормальное: $\Phi(x)$')
axes[1].plot(x, logistic.cdf(x), color=color_log, lw=2.5, linestyle='--', label=r'Логистическое: $\Lambda(x)$')
axes[1].set_title('Кумулятивная функция (CDF)')
axes[1].set_xlabel('$x$')
axes[1].set_ylabel('Вероятность')

# --- График 3: Квантильная функция (PPF) ---
axes[2].plot(p, norm.ppf(p), color=color_norm, lw=2.5, label='Probit')
axes[2].plot(p, logistic.ppf(p), color=color_log, lw=2.5, linestyle='--', label='Logit')
axes[2].set_title('Квантильная функция (PPF)')
axes[2].set_xlabel('Вероятность $p$')
axes[2].set_ylabel('Значение ($x$)')
axes[2].set_ylim(-6, 6)

# --- Общие настройки для всех графиков в цикле ---
for ax in axes:
    # Отрисовка осей, проходящих через ноль
    ax.axhline(0, color='black', linewidth=1, alpha=0.3)
    ax.axvline(0, color='black', linewidth=1, alpha=0.3)
    # Настройка легенды
    ax.legend(frameon=True, shadow=True, loc='best')

# Автоматически выравнивает отступы между графиками, чтобы ничего не наезжало
plt.tight_layout()
plt.show()

<Figure size 4800x1500 with 3 Axes>

# Оценивание и интерпретация коэффициентов

## approve equation #1 (probit)

Для датасета `loanapp` рассмотрим probit-регрессию __approve на appinc, mortno, unem, dep, male, married, yjob, self__

Спецификация: 

$$
P(approve=1)=\Phi(\beta_0+\beta_1appinc+\beta_2mortno+\beta_3unem+\beta_4dep+\beta_5male+\beta_6married+\beta_7yjob+\beta_8self)
$$

Альтернативная спецификация:

$$
probit(P(approve=1))=\beta_0+\beta_1appinc+\beta_2mortno+\beta_3unem+\beta_4dep+\beta_5male+\beta_6married+\beta_7yjob+\beta_8self
$$


Оцените модель на данных и укажите коэффициенты подогнанной модели. **Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [6]:
#| echo: false
#| warning: false
#| output: asis

formula_app = 'approve ~ appinc + mortno + unem + dep + male + married + yjob + self'
mod_probit_app = probit(formula_app, data=loanapp).fit(disp=0)
print_coefs(mod_probit_app, digits=3)

| Переменная   |   Оценка (MLE) |
|:-------------|---------------:|
| Intercept    |          1.142 |
| appinc       |         -0.001 |
| mortno       |          0.407 |
| unem         |         -0.031 |
| dep          |         -0.083 |
| male         |          0.02  |
| married      |          0.221 |
| yjob         |         -0.001 |
| self         |         -0.158 |

Дайте интерпретацию коэффициентам модели.

## approve equation #2 (logit)

Для датасета `loanapp` рассмотрим logit-регрессию __approve на appinc, mortno, unem, dep, male, married, yjob, self__

Спецификация: 

$$
P(approve=1)=\Lambda(\beta_0+\beta_1appinc+\beta_2mortno+\beta_3unem+\beta_4dep+\beta_5male+\beta_6married+\beta_7yjob+\beta_8self)
$$

Альтернативная спецификация:

$$
logit(P(approve=1))=\beta_0+\beta_1appinc+\beta_2mortno+\beta_3unem+\beta_4dep+\beta_5male+\beta_6married+\beta_7yjob+\beta_8self
$$

Здесь 

$$
logit(P(approve=1)) = \log\frac{P(approve=1)}{1-P(approve=1)} = \log\frac{P(approve=1)}{P(approve=0)}
$$

Оцените модель на данных и укажите коэффициенты подогнанной модели. **Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [7]:
#| echo: false
#| warning: false
#| output: asis

mod_logit_app = logit(formula_app, data=loanapp).fit(disp=0)
print_coefs(mod_logit_app, digits=3)

| Переменная   |   Оценка (MLE) |
|:-------------|---------------:|
| Intercept    |          1.931 |
| appinc       |         -0.001 |
| mortno       |          0.787 |
| unem         |         -0.055 |
| dep          |         -0.161 |
| male         |          0.03  |
| married      |          0.425 |
| yjob         |         -0.006 |
| self         |         -0.28  |

Дайте интерпретацию коэффициентам модели.

## labour force equation #1 (probit)

Для датасета `TableF5-1` рассмотрим probit-регрессию __LFP на WA, I(WA ** 2), WE, KL6, K618, CIT, UN, np.log(FAMINC)__

Спецификация: 

$$
P(LFP=1)=\Phi(\beta_0+\beta_1WA+\beta_2WA^2+\beta_3WE+\beta_4KL6+\beta_5K618+\beta_6CIT+\beta_7UN+\beta_8\log(FAMINC))
$$

Альтернативная спецификация:

$$
probit(P(LFP=1))=\beta_0+\beta_1WA+\beta_2WA^2+\beta_3WE+\beta_4KL6+\beta_5K618+\beta_6CIT+\beta_7UN+\beta_8\log(FAMINC)
$$



Оцените модель на данных и укажите коэффициенты подогнанной модели. **Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [8]:
#| echo: false
#| warning: false
#| output: asis

formula_lfp = 'LFP ~ WA + I(WA**2) + WE + KL6 + K618 + CIT + UN + np.log(FAMINC)'
mod_probit_lfp = probit(formula_lfp, data=mroz_Greene).fit(disp=0)
print_coefs(mod_probit_lfp, digits=3)

| Переменная     |   Оценка (MLE) |
|:---------------|---------------:|
| Intercept      |         -2.005 |
| WA             |          0.008 |
| I(WA ** 2)     |         -0.001 |
| WE             |          0.109 |
| KL6            |         -0.851 |
| K618           |         -0.063 |
| CIT            |         -0.128 |
| UN             |         -0.011 |
| np.log(FAMINC) |          0.2   |

Дайте интерпретацию коэффициентам модели.

## labour force equation #2 (logit)

Для датасета `TableF5-1` рассмотрим logit-регрессию __LFP на WA, I(WA ** 2), WE, KL6, K618, CIT, UN, np.log(FAMINC)__

Спецификация: 

$$
P(LFP=1)=\Lambda(\beta_0+\beta_1WA+\beta_2WA^2+\beta_3WE+\beta_4KL6+\beta_5K618+\beta_6CIT+\beta_7UN+\beta_8\log(FAMINC))
$$

Альтернативная спецификация:

$$
logit(LFP)=\beta_0+\beta_1WA+\beta_2WA^2+\beta_3WE+\beta_4KL6+\beta_5K618+\beta_6CIT+\beta_7UN+\beta_8\log(FAMINC)
$$

Здесь 

$$
logit(P(LFP=1)) = \log\frac{P(LFP=1)}{1-P(LFP=1)} = \log\frac{P(LFP=1)}{P(LFP=0)}
$$

Оцените модель на данных и укажите коэффициенты подогнанной модели. **Ответ округлите до 3-х десятичных знаков.**

Ответ:

In [9]:
#| echo: false
#| warning: false
#| output: asis

mod_logit_lfp = logit(formula_lfp, data=mroz_Greene).fit(disp=0)
print_coefs(mod_logit_lfp, digits=3)

| Переменная     |   Оценка (MLE) |
|:---------------|---------------:|
| Intercept      |         -3.241 |
| WA             |          0.007 |
| I(WA ** 2)     |         -0.001 |
| WE             |          0.18  |
| KL6            |         -1.414 |
| K618           |         -0.104 |
| CIT            |         -0.217 |
| UN             |         -0.018 |
| np.log(FAMINC) |          0.333 |

Дайте интерпретацию коэффициентам модели.

# z-test

## approve equation #1 (probit)

Для датасета `loanapp` рассмотрим probit-регрессию __approve на appinc, mortno, unem, dep, male, married, yjob, self__

Подгоните модель на данных и приведите результаты $z$-теста

Ответ:

In [10]:
#| echo: false
#| warning: false
#| output: asis

sign_level = 0.10
print_z_test(mod_probit_app, sign_level=sign_level, digits=4)

text_info = f"Модель была подогнана по {int(mod_probit_app.nobs)} наблюдениям. <span style='color: blue'>Уровень значимости {int(sign_level*100)}%</span>"
display(Markdown(f"\n\n{text_info}\n\n"))

|           |   Coef. |   Std.Err. |   z-value |   p-value | Signif.   |
|:----------|--------:|-----------:|----------:|----------:|:----------|
| Intercept |  1.1418 |     0.1086 |   10.5122 |    0      |           |
| appinc    | -0.0005 |     0.0004 |   -1.3752 |    0.1691 |           |
| mortno    |  0.4071 |     0.0866 |    4.7026 |    0      | ***       |
| unem      | -0.0308 |     0.0161 |   -1.9086 |    0.0563 | *         |
| dep       | -0.0828 |     0.0352 |   -2.3546 |    0.0185 | **        |
| male      |  0.02   |     0.0995 |    0.2009 |    0.8408 |           |
| married   |  0.2208 |     0.0865 |    2.5522 |    0.0107 | **        |
| yjob      | -0.0007 |     0.0349 |   -0.02   |    0.984  |           |
| self      | -0.1583 |     0.1067 |   -1.4826 |    0.1382 |           |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 1971 наблюдениям. <span style='color: blue'>Уровень значимости 10%</span>



Вычислите необходимое критическое значение $z$-теста. **Ответ округлите до 3-х десятичных знаков.**

In [11]:
#| echo: false
#| warning: false
#| output: asis

sign_level = 0.10
z_cr = norm.isf(sign_level/2)  # Критическое значение для двустороннего теста
display_answer(f"$z_{{crit}} = {z_cr:.3f}$", title="Критическое значение z-теста")



::: {.callout-note title="Критическое значение z-теста" icon="true"}
$z_{crit} = 1.645$
:::



Какие коэффициенты значимы?

In [12]:
#| echo: false
#| warning: false
#| output: asis

print_significant_coeffs(mod_probit_app, sign_level=sign_level)



::: {.callout-note title="Коэффициенты, значимые на уровне 10%" icon="true"}
`mortno`, `unem`, `dep`, `married`
:::



## approve equation #2 (logit)

Для датасета `loanapp` рассмотрим logit-регрессию __approve на appinc, mortno, unem, dep, male, married, yjob, self__

Подгоните модель на данных и приведите результаты $z$-теста

Ответ:

In [13]:
#| echo: false
#| warning: false
#| output: asis

sign_level = 0.05
print_z_test(mod_logit_app, sign_level=sign_level, digits=4)

text_info = f"Модель была подогнана по {int(mod_logit_app.nobs)} наблюдениям. <span style='color: blue'>Уровень значимости {int(sign_level*100)}%</span>"
display(Markdown(f"\n\n{text_info}\n\n"))

|           |   Coef. |   Std.Err. |   z-value |   p-value | Signif.   |
|:----------|--------:|-----------:|----------:|----------:|:----------|
| Intercept |  1.9315 |     0.1993 |    9.6891 |    0      |           |
| appinc    | -0.001  |     0.0007 |   -1.4717 |    0.1411 |           |
| mortno    |  0.7868 |     0.1721 |    4.5714 |    0      | ***       |
| unem      | -0.0549 |     0.0294 |   -1.8661 |    0.062  | *         |
| dep       | -0.1608 |     0.0647 |   -2.4861 |    0.0129 | **        |
| male      |  0.03   |     0.1859 |    0.1612 |    0.8719 |           |
| married   |  0.4246 |     0.1624 |    2.6145 |    0.0089 | ***       |
| yjob      | -0.0065 |     0.0651 |   -0.0993 |    0.9209 |           |
| self      | -0.2804 |     0.1967 |   -1.4257 |    0.1539 |           |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 1971 наблюдениям. <span style='color: blue'>Уровень значимости 5%</span>



Вычислите необходимое критическое значение $z$-теста. **Ответ округлите до 3-х десятичных знаков.**

In [14]:
#| echo: false
#| warning: false
#| output: asis

sign_level = 0.05
z_cr = norm.isf(sign_level/2)  # Критическое значение для двустороннего теста

display_answer(f"$z_{{crit}} = {z_cr:.3f}$", title="Критическое значение z-теста")



::: {.callout-note title="Критическое значение z-теста" icon="true"}
$z_{crit} = 1.960$
:::



Какие коэффициенты значимы?

In [15]:
#| echo: false
#| warning: false
#| output: asis

print_significant_coeffs(mod_logit_app, sign_level=sign_level)



::: {.callout-note title="Коэффициенты, значимые на уровне 5%" icon="true"}
`mortno`, `dep`, `married`
:::



## labour force equation #1 (probit)

Для датасета `TableF5-1` рассмотрим probit-регрессию __LFP на WA, I(WA ** 2), WE, KL6, K618, CIT, UN, np.log(FAMINC)__

Подгоните модель на данных и приведите результаты z-теста

Ответ:

In [16]:
#| echo: false
#| warning: false
#| output: asis

sign_level = 0.10
print_z_test(mod_probit_lfp, sign_level=sign_level, digits=4)

text_info = f"Модель была подогнана по {int(mod_probit_lfp.nobs)} наблюдениям. <span style='color: blue'>Уровень значимости {int(sign_level*100)}%</span>"
display(Markdown(f"\n\n{text_info}\n\n"))

|                |   Coef. |   Std.Err. |   z-value |   p-value | Signif.   |
|:---------------|--------:|-----------:|----------:|----------:|:----------|
| Intercept      | -2.0046 |     1.7055 |   -1.1754 |    0.2398 |           |
| WA             |  0.0076 |     0.0703 |    0.1084 |    0.9137 |           |
| I(WA ** 2)     | -0.0005 |     0.0008 |   -0.6538 |    0.5132 |           |
| WE             |  0.1088 |     0.0241 |    4.514  |    0      | ***       |
| KL6            | -0.8513 |     0.1146 |   -7.4277 |    0      | ***       |
| K618           | -0.0632 |     0.0415 |   -1.5206 |    0.1284 |           |
| CIT            | -0.1277 |     0.1074 |   -1.1891 |    0.2344 |           |
| UN             | -0.0106 |     0.0157 |   -0.6785 |    0.4975 |           |
| np.log(FAMINC) |  0.1996 |     0.1046 |    1.9081 |    0.0564 | *         |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 753 наблюдениям. <span style='color: blue'>Уровень значимости 10%</span>



Вычислите необходимое критическое значение $z$-теста. **Ответ округлите до 3-х десятичных знаков.**

In [17]:
#| echo: false
#| warning: false
#| output: asis

sign_level = 0.10
z_cr = norm.isf(sign_level/2)  # Критическое значение для двустороннего теста
display_answer(f"$z_{{crit}} = {z_cr:.3f}$", title="Критическое значение z-теста")



::: {.callout-note title="Критическое значение z-теста" icon="true"}
$z_{crit} = 1.645$
:::



Какие коэффициенты значимы?

In [18]:
#| echo: false
#| warning: false
#| output: asis

print_significant_coeffs(mod_probit_lfp, sign_level=sign_level)



::: {.callout-note title="Коэффициенты, значимые на уровне 10%" icon="true"}
`WE`, `KL6`, `np.log(FAMINC)`
:::



## labour force equation #2 (logit)

Для датасета `TableF5-1` рассмотрим logit-регрессию __LFP на WA, I(WA ** 2), WE, KL6, K618, CIT, UN, np.log(FAMINC)__

Подгоните модель на данных и приведите результаты $z$-теста

Ответ:

In [19]:
#| echo: false
#| warning: false
#| output: asis

sign_level = 0.05
print_z_test(mod_logit_lfp, sign_level=sign_level, digits=4)

text_info = f"Модель была подогнана по {int(mod_logit_lfp.nobs)} наблюдениям. <span style='color: blue'>Уровень значимости {int(sign_level*100)}%</span>"
display(Markdown(f"\n\n{text_info}\n\n"))

|                |   Coef. |   Std.Err. |   z-value |   p-value | Signif.   |
|:---------------|--------:|-----------:|----------:|----------:|:----------|
| Intercept      | -3.2407 |     2.8337 |   -1.1436 |    0.2528 |           |
| WA             |  0.007  |     0.1159 |    0.0602 |    0.952  |           |
| I(WA ** 2)     | -0.0008 |     0.0013 |   -0.6061 |    0.5444 |           |
| WE             |  0.18   |     0.0404 |    4.4535 |    0      | ***       |
| KL6            | -1.4138 |     0.1987 |   -7.1152 |    0      | ***       |
| K618           | -0.1042 |     0.0687 |   -1.5166 |    0.1294 |           |
| CIT            | -0.2165 |     0.1765 |   -1.2267 |    0.2199 |           |
| UN             | -0.0176 |     0.0258 |   -0.6812 |    0.4957 |           |
| np.log(FAMINC) |  0.3331 |     0.1729 |    1.9272 |    0.054  | *         |

*Signif. codes:* `***` $p<0.01$, `**` $p<0.05$, `*` $p<0.1$



Модель была подогнана по 753 наблюдениям. <span style='color: blue'>Уровень значимости 5%</span>



Вычислите необходимое критическое значение $z$-теста. **Ответ округлите до 3-х десятичных знаков.**

In [20]:
#| echo: false
#| warning: false
#| output: asis

sign_level = 0.05
z_cr = norm.isf(sign_level/2)  # Критическое значение для двустороннего теста
display_answer(f"$z_{{crit}} = {z_cr:.3f}$", title="Критическое значение z-теста")



::: {.callout-note title="Критическое значение z-теста" icon="true"}
$z_{crit} = 1.960$
:::



Какие коэффициенты значимы?

In [21]:
#| echo: false
#| warning: false
#| output: asis

print_significant_coeffs(mod_logit_lfp, sign_level=sign_level)



::: {.callout-note title="Коэффициенты, значимые на уровне 5%" icon="true"}
`WE`, `KL6`
:::



# LR-тест: значимость регрессии

## approve equation #1 (probit)

Для датасета `loanapp` рассмотрим probit-регрессию __approve на appinc, unem, male, yjob, self__

In [22]:
#| echo: false
#| warning: false
#| output: asis

sign_level = 0.10
mod_probit_app_short = probit('approve ~ appinc + unem + male + yjob + self', data=loanapp).fit(disp=0)

text = f"""
- число наблюдений: {int(mod_probit_app_short.nobs)}
- логарифм функции правдоподобия: $\log L = {mod_probit_app_short.llf:.4f}$
- логарифм функции правдоподобия для регрессии на константу: $\log L_{{null}} = {mod_probit_app_short.llnull:.4f}$
"""
display(Markdown(text))



- число наблюдений: 1971
- логарифм функции правдоподобия: $\log L = -733.7270$
- логарифм функции правдоподобия для регрессии на константу: $\log L_{null} = -737.9793$


Тестируется значимость регрессии, т.е. гипотеза $H_0:\beta_{appinc}=\beta_{unem}=\beta_{male}=\beta_{yjob}=\beta_{self}=0$.
<span style="color: blue">Уровень значимости 10%.</span>

Вычислите тестовую статистику. **Ответ округлите до 3-х десятичных знаков.**

In [23]:
#| echo: false
#| warning: false
#| output: asis

lr_stat = mod_probit_app_short.llr
display_answer(f"$LR = {lr_stat:.3f}$", title="LR Статистика")



::: {.callout-note title="LR Статистика" icon="true"}
$LR = 8.505$
:::



Вычислите критическое значение. **Ответ округлите до 3-х десятичных знаков.**

In [24]:
#| echo: false
#| warning: false
#| output: asis

chi2_cr = chi2.ppf(1 - 0.10, df=mod_probit_app_short.df_model)
display_answer(f"$\chi^2_{{crit}} = {chi2_cr:.3f}$", title="Критическое значение")

if lr_stat > chi2_cr:
    display_answer("Регрессия **значима**.", theme="tip", title="Вывод")
else:
    display_answer("Регрессия **незначима**.", theme="important", title="Вывод")

display(Markdown("**Какие коэффициенты значимы?**"))
print_significant_coeffs(mod_probit_app_short, sign_level=0.10)



::: {.callout-note title="Критическое значение" icon="true"}
$\chi^2_{crit} = 9.236$
:::





::: {.callout-important title="Вывод" icon="true"}
Регрессия **незначима**.
:::



**Какие коэффициенты значимы?**



::: {.callout-note title="Коэффициенты, значимые на уровне 10%" icon="true"}
`unem`
:::



## labour force equation #1 (probit)

Для датасета `TableF5-1` рассмотрим несколько probit-регрессий.

In [25]:
#| echo: false
#| warning: false
#| output: asis

sign_level = 0.10
mods = []
f1 = 'LFP ~ WA + I(WA ** 2) + WE + KL6 + K618 + CIT + UN + np.log(FAMINC)'
f2 = 'LFP ~ WA + I(WA ** 2) + CIT + UN + np.log(FAMINC)'
f3 = 'LFP ~ WA + I(WA ** 2) + CIT + UN'
f4 = 'LFP ~ CIT + UN'
mods.append(probit(f1, data=mroz_Greene).fit(disp=0))
mods.append(probit(f2, data=mroz_Greene).fit(disp=0))
mods.append(probit(f3, data=mroz_Greene).fit(disp=0))
mods.append(probit(f4, data=mroz_Greene).fit(disp=0))

res_list = []
for i, m in enumerate(mods):
    lr = m.llr
    cr = chi2.ppf(1 - sign_level, df=m.df_model)
    sign = 'Значима' if lr > cr else 'Незначима'
    res_list.append({
        'Регрессия': i + 1,
        'LR stat': round(lr, 3),
        'Критическое значение': round(cr, 3),
        'Значимость': sign
    })

df_lr = pd.DataFrame(res_list)
display(Markdown(df_lr.to_markdown(index=False)))

|   Регрессия |   LR stat |   Критическое значение | Значимость   |
|------------:|----------:|-----------------------:|:-------------|
|           1 |   105.066 |                 13.362 | Значима      |
|           2 |    25.299 |                  9.236 | Значима      |
|           3 |    10.44  |                  7.779 | Значима      |
|           4 |     0.62  |                  4.605 | Незначима    |

# LR-тест: совместная значимость

## labour force equation #1 (probit)

Для датасета `TableF5-1` рассмотрим probit-регрессию __LFP на WA, WA ** 2, WE, KL6, K618, CIT, UN, np.log(FAMINC)__

In [26]:
#| echo: false
#| warning: false
#| output: asis

sign_level = 0.10
mod_full = probit('LFP ~ WA + I(WA**2) + WE + KL6 + K618 + CIT + UN + np.log(FAMINC)', data=mroz_Greene).fit(disp=0)
mod_r1 = probit('LFP ~ WE + KL6 + K618 + CIT + UN + np.log(FAMINC)', data=mroz_Greene).fit(disp=0)
mod_r2 = probit('LFP ~ WA + I(WA**2) + WE + KL6 + np.log(FAMINC)', data=mroz_Greene).fit(disp=0)

display(Markdown(f"<span style='color: blue'>Уровень значимости {int(sign_level*100)}%.</span>"))

<span style='color: blue'>Уровень значимости 10%.</span>

### Гипотеза 1

Тестируется значимость влияния возраста, т.е. гипотеза $H_0:\beta_{WA}=\beta_{WA^2}=0$ по LR-тесту.

In [27]:
#| echo: false
#| warning: false
#| output: asis

lr_stat = 2 * (mod_full.llf - mod_r1.llf)
df_diff = mod_r1.df_resid - mod_full.df_resid
chi2_cr = chi2.ppf(1 - sign_level, df=df_diff)

text = f"**LR-статистика:** {lr_stat:.3f} \n\n**Критическое значение:** {chi2_cr:.3f}"
display(Markdown(text))

if lr_stat > chi2_cr:
    display_answer("Влияние возраста **значимо**.", theme="tip", title="Вывод")
else:
    display_answer("Влияние возраста **незначимо**.", theme="important", title="Вывод")

**LR-статистика:** 26.718 

**Критическое значение:** 4.605



::: {.callout-tip title="Вывод" icon="true"}
Влияние возраста **значимо**.
:::



### Гипотеза 2

Тестируется совместная значимость влияния **K618, CIT, UN**, т.е. гипотеза $H_0:\beta_{K618}=\beta_{CIT}=\beta_{UN}=0$ по LR-тесту.

In [28]:
#| echo: false
#| warning: false
#| output: asis

lr_stat = 2 * (mod_full.llf - mod_r2.llf)
df_diff = mod_r2.df_resid - mod_full.df_resid
chi2_cr = chi2.ppf(1 - sign_level, df=df_diff)

text = f"**LR-статистика:** {lr_stat:.3f} \n\n**Критическое значение:** {chi2_cr:.3f}"
display(Markdown(text))

if lr_stat > chi2_cr:
    display_answer("Влияние переменных **значимо**.", theme="tip", title="Вывод")
else:
    display_answer("Влияние переменных **незначимо**.", theme="important", title="Вывод")

**LR-статистика:** 4.712 

**Критическое значение:** 6.251



::: {.callout-important title="Вывод" icon="true"}
Влияние переменных **незначимо**.
:::



# Тест Вальда: совместная значимость

## swiss labour force equation #1

Для датасета `SwissLabor` рассмотрим logit-регрессию __participation на income, I(income ** 2), age, I(age ** 2), youngkids, oldkids, foreign__

Оцените регрессию на данных и тестируйте следующие гипотезы, используя $\chi^2$-статистику Вальда.

In [29]:
#| echo: false
#| warning: false
#| output: asis

sign_level = 0.05
formula_swiss = 'participation_num ~ income + I(income**2) + age + I(age**2) + youngkids + oldkids + foreign'
mod_swiss = logit(formula_swiss, data=SwissLabor).fit(disp=0)

display(Markdown(f"<span style='color: blue'>Уровень значимости {int(sign_level*100)}%.</span>"))

<span style='color: blue'>Уровень значимости 5%.</span>

### Гипотеза 1

Тестируется значимость влияния дохода, т.е. гипотеза $H_0:\beta_{income}=\beta_{income^2}=0$.

In [30]:
#| echo: false
#| warning: false
#| output: asis

wald_test = mod_swiss.wald_test("income = 0, I(income ** 2) = 0")
wald_stat = wald_test.statistic.item()
p_val = wald_test.pvalue.item()
chi2_cr = chi2.ppf(1 - sign_level, df=2)

text = f"**Статистика Вальда:** {wald_stat:.3f} \n\n**P-значение:** {p_val:.4f} \n\n**Критическое значение:** {chi2_cr:.3f}"
display(Markdown(text))

if p_val < sign_level:
    display_answer("Влияние дохода **значимо**.", theme="tip", title="Вывод")
else:
    display_answer("Влияние дохода **незначимо**.", theme="important", title="Вывод")

**Статистика Вальда:** 24.441 

**P-значение:** 0.0000 

**Критическое значение:** 5.991



::: {.callout-tip title="Вывод" icon="true"}
Влияние дохода **значимо**.
:::



### Гипотеза 2

Тестируется значимость влияния числа детей, т.е. гипотеза $H_0:\beta_{youngkids}=\beta_{oldkids}=0$.

In [31]:
#| echo: false
#| warning: false
#| output: asis

wald_test = mod_swiss.wald_test("youngkids = 0, oldkids = 0")
wald_stat = wald_test.statistic.item()
p_val = wald_test.pvalue.item()
chi2_cr = chi2.ppf(1 - sign_level, df=2)

text = f"**Статистика Вальда:** {wald_stat:.3f} \n\n**P-значение:** {p_val:.4f} \n\n**Критическое значение:** {chi2_cr:.3f}"
display(Markdown(text))

if p_val < sign_level:
    display_answer("Влияние числа детей **значимо**.", theme="tip", title="Вывод")
else:
    display_answer("Влияние числа детей **незначимо**.", theme="important", title="Вывод")

**Статистика Вальда:** 48.420 

**P-значение:** 0.0000 

**Критическое значение:** 5.991



::: {.callout-tip title="Вывод" icon="true"}
Влияние числа детей **значимо**.
:::



### Гипотеза 3

Тестируется значимость влияния возраста, т.е. гипотеза $H_0:\beta_{age}=\beta_{age^2}=0$.

In [32]:
#| echo: false
#| warning: false
#| output: asis

wald_test = mod_swiss.wald_test("age = 0, I(age ** 2) = 0")
wald_stat = wald_test.statistic.item()
p_val = wald_test.pvalue.item()
chi2_cr = chi2.ppf(1 - sign_level, df=2)

text = f"**Статистика Вальда:** {wald_stat:.3f} \n\n**P-значение:** {p_val:.4f} \n\n**Критическое значение:** {chi2_cr:.3f}"
display(Markdown(text))

if p_val < sign_level:
    display_answer("Влияние возраста **значимо**.", theme="tip", title="Вывод")
else:
    display_answer("Влияние возраста **незначимо**.", theme="important", title="Вывод")

**Статистика Вальда:** 58.911 

**P-значение:** 0.0000 

**Критическое значение:** 5.991



::: {.callout-tip title="Вывод" icon="true"}
Влияние возраста **значимо**.
:::

